# 03 - Neighbourhood Graphs

Following source `generate_ngraphs.py` — builds 5 spatial graphs:
1. **Tessellation graph** — fuzzy contiguity (buffer=1e-6) + self-weight
2. **Building graph** — fuzzy contiguity (buffer=0.25) + self-weight
3. **Street graph** — Queen contiguity on edges + self-weight
4. **Enclosure graph** — Queen contiguity on enclosures + self-weight
5. **Node graph** — from NetworkX primal graph of streets

In [1]:
import os
os.chdir(os.path.join(os.path.dirname(os.getcwd()), "") if os.path.basename(os.getcwd()) == "notebooks" else ".")
print("Working dir:", os.getcwd())

Working dir: c:\urban-form\urban_taxonomy


In [2]:
import gc
import geopandas as gpd
import momepy as mm
from libpysal.graph import Graph
from pathlib import Path

OUT = Path("outputs/03_graphs")
OUT.mkdir(parents=True, exist_ok=True)

## Load data

In [3]:
buildings    = gpd.read_file("outputs/01_preprocessed/buildings.gpkg")
streets      = gpd.read_file("outputs/01_preprocessed/roads.gpkg")
tessellation = gpd.read_parquet("outputs/02_tessellation/tessellation.parquet")
enclosures   = gpd.read_parquet("outputs/02_tessellation/enclosures.parquet")

print(f"Buildings: {len(buildings):,}, Streets: {len(streets):,}")
print(f"Tessellation: {len(tessellation):,}, Enclosures: {len(enclosures):,}")

Buildings: 621,480, Streets: 17,590
Tessellation: 625,317, Enclosures: 10,764


## 1. Tessellation graph (fuzzy contiguity)

In [4]:
tess_graph = Graph.build_fuzzy_contiguity(tessellation, buffer=1e-6).assign_self_weight()
tess_graph.to_parquet(OUT / "tessellation_graph.parquet")
print(f"Tessellation graph: {tess_graph.n} nodes")
del tess_graph; gc.collect()

Tessellation graph: 625317 nodes


0

## 2. Building graph (fuzzy contiguity, buffer > simplification tolerance)

In [5]:
bldg_graph = Graph.build_fuzzy_contiguity(buildings, buffer=0.25).assign_self_weight()
bldg_graph.to_parquet(OUT / "building_graph.parquet")
print(f"Building graph: {bldg_graph.n} nodes")
del bldg_graph; gc.collect()

Building graph: 621480 nodes


0

## 3. Street graph (Queen contiguity on edges)

In [6]:
str_graph = Graph.build_contiguity(streets, rook=False).assign_self_weight()
str_graph.to_parquet(OUT / "street_graph.parquet")
print(f"Street graph: {str_graph.n} nodes")
del str_graph; gc.collect()

Street graph: 17590 nodes


0

## 4. Enclosure graph (Queen contiguity)

In [7]:
encl_graph = Graph.build_contiguity(enclosures, rook=False).assign_self_weight()
encl_graph.to_parquet(OUT / "enclosure_graph.parquet")
print(f"Enclosure graph: {encl_graph.n} nodes")
del encl_graph; gc.collect()

Enclosure graph: 10764 nodes


0

## 5. Node graph (from NX primal street graph)

In [8]:
nx_graph = mm.gdf_to_nx(streets, preserve_index=True)
nx_graph = mm.node_degree(nx_graph)
_, _, w = mm.nx_to_gdf(nx_graph, spatial_weights=True)

nodes_graph = Graph.from_W(w)
nodes_graph.to_parquet(OUT / "nodes_graph.parquet")
print(f"Nodes graph: {nodes_graph.n} nodes")

del nx_graph, w, nodes_graph; gc.collect()
print("All 5 graphs saved to", OUT)

c:\urban-form\urban_taxonomy\.venv\Lib\site-packages\libpysal\weights\weights.py:1683: UserWarning: The weights matrix is not fully connected: 
 There are 3698 disconnected components.
  w = W(neighbors, weights, ids, silence_warnings=silence_warnings)


Nodes graph: 18293 nodes
All 5 graphs saved to outputs\03_graphs
